In [1]:
import pandas as pd
import numpy as np 
from typing import List
from sklearn.preprocessing import StandardScaler

def process() -> pd.DataFrame:
    sheets = pd.read_excel("/kaggle/input/boroondara2006/Scats Data October 2006.xls", sheet_name=None)
    removed_days = sheets["Notes"]["Unnamed: 1"].dropna().to_list()[5:]
    removed_days = [int(i) for i in removed_days]
    
    df_info = sheets["Summary Of Data"].loc[2:]
    df_info.columns = df_info.iloc[0]
    df_info = df_info[1:]
    df_info["SCATS Number"] = df_info["SCATS Number"].ffill()
    
    df_info.dropna(axis=1, inplace=True)
    df_info = df_info.convert_dtypes().infer_objects()
    
    df_info_filtered = df_info[df_info["Total"].between(7, 31)]
    df = sheets["Data"]
    df.columns = pd.Series(df.loc[0])
    
    # I love formatting
    df = df.loc[1:]
    df.loc[:, 'Date'] = pd.to_datetime(df['Date']).dt.date
    df = df.convert_dtypes()
    
    # filter good locations
    df = df.loc[df["Location"].isin(df_info_filtered["Location"])]
    
    # add location identifier: SCARS Number + VicRoads Internal
    df.loc[:, "Identifier"] = df["SCATS Number"].astype(str) + " - " + df["HF VicRoads Internal"].astype(str)
    df.columns = df.columns.str.strip()
    df = df.fillna(0)
    df.reset_index(drop=True)
    df.to_csv("/kaggle/working/scats.csv")
    return df
    
def make_flow(df: pd.DataFrame) -> pd.DataFrame:
    vcols = [f"V{str(i).zfill(2)}" for i in range(96)]
    DAY_LENGTH = 96 # intervals per day
    MONTH_LENGTH = 31 # days in oct
    d = {}
    for id, group in df.groupby(by="Identifier"):
        # identifier + flow
        flow = list(group[vcols].fillna(0).to_numpy().flatten())
        while len(flow) < DAY_LENGTH * MONTH_LENGTH: flow.append(0)
        d[id] = flow
    d = {i: np.array(d[i]) for i in d}
    flow = pd.DataFrame(d)
    return flow

def make_windows(flow: pd.DataFrame) -> np.ndarray:
    data = flow.to_numpy().astype(float)
    windows = []
    DAY_LENGTH = 96 # intervals per day
    WEEK_LENGTH = DAY_LENGTH * 7  # 1 week
    WINDOW_LENGTH = WEEK_LENGTH
    for i in range(WINDOW_LENGTH, len(data) + 1):
        windows.append(data[i - WINDOW_LENGTH : i])
    windows = np.array(windows)
    NUM_WINDOW, _, NUM_LOCATION = windows.shape
    for name, val in zip(["Number of windows", "Window Length", "Number of locations"], windows.shape):
        print(f"{name}: {val}")
    return windows

def make_dataset(windows: np.ndarray) -> List[np.ndarray]:
    NUM_WINDOW, WINDOW_LENGTH, NUM_LOCATION = windows.shape
    HORIZON = 24  # assuming hourly data
    # Input is all but last HORIZON steps
    X = np.array(windows[:, :-HORIZON, :])  # shape: (num_windows, WINDOW_LENGTH - HORIZON, NUM_LOCATIONS)
    
    # Output is the next HORIZON steps
    y = np.array(windows[:, -HORIZON:, :])  # shape: (num_windows, HORIZON, NUM_LOCATIONS)
    
    train_size = int(NUM_WINDOW * 0.7)
    val_size = int(NUM_WINDOW * 0.15)
    X_train = X[:train_size]
    X_val   = X[train_size : train_size + val_size]
    X_test  = X[train_size + val_size :]
    y_train = y[:train_size]
    y_val   = y[train_size : train_size + val_size]
    y_test  = y[train_size + val_size :]
    print("Shapes: ")
    print("X_train:", X_train.shape)
    print("X_val:", X_val.shape)
    print("X_test:", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_val:", y_val.shape)
    print("y_test:", y_test.shape) 
    return [X_train, X_val, X_test, y_train, y_val, y_test]


df = process()
flow = make_flow(df)
windows = make_windows(flow)
scats = {}
for i, v in enumerate(flow.columns):
    z = v[:4]
    if scats.get(z) is None:
        scats[z] = [i]
    else:
        scats[z].append(i)

print(df)
print(flow)
print(windows)
print(scats)

Number of windows: 2305
Window Length: 672
Number of locations: 137
0    SCATS Number                         Location CD_MELWAY  NB_LATITUDE  \
1            0970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
2            0970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
3            0970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
4            0970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
5            0970  WARRIGAL_RD N of HIGH STREET_RD   060 G10    -37.86703   
...           ...                              ...       ...          ...   
4188         4821      VICTORIA_ST W OF BURNLEY_ST   002HF02    -37.81296   
4189         4821      VICTORIA_ST W OF BURNLEY_ST   002HF02    -37.81296   
4190         4821      VICTORIA_ST W OF BURNLEY_ST   002HF02    -37.81296   
4191         4821      VICTORIA_ST W OF BURNLEY_ST   002HF02    -37.81296   
4192         4821      VICTORIA_ST W OF BURNLEY_ST   002HF02    -37.81296   

0     N

In [2]:
# make dirs
import os
k = "/kaggle/working"
for d in ["model/", "images/"]:
    path = os.path.join(k, d)
    if not os.path.exists(path):
        print(f"Created directory {str(path)}")
        os.mkdir(path)
        

Created directory /kaggle/working/model/
Created directory /kaggle/working/images/


In [3]:
"""
Train the NN model.
"""
from sklearn import metrics
import argparse
import keras
import numpy as np
import os
from keras.layers import LSTM, GRU, Dense, Input, Dropout, TimeDistributed, RepeatVector
from keras.models import Sequential
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.saving import load_model
from keras.utils import plot_model
# from process import windows, make_dataset
from sklearn.preprocessing import StandardScaler
from typing import Any, Dict
import pandas as pd

def fit_scaler(A: np.ndarray) -> StandardScaler:
    return StandardScaler().fit(A.reshape(-1, 1))

def normalize(scaler: StandardScaler, A: np.ndarray) -> np.ndarray:
    return scaler.transform(A.reshape(-1, 1)).reshape(A.shape)

# for predictions
def denormalize(scaler: StandardScaler, A: np.ndarray) -> np.ndarray:
    return scaler.inverse_transform(A.reshape(-1, 1)).reshape(A.shape)

X_train, X_val, X_test, y_train, y_val, y_test = make_dataset(windows)
scaler = fit_scaler(X_train)
X_train_n, X_val_n, X_test_n = normalize(scaler, X_train), normalize(scaler, X_val), normalize(scaler, X_test)
y_train_n, y_val_n, y_test_n = normalize(scaler, y_train), normalize(scaler, y_val), normalize(scaler, y_test)

NUM_WINDOW, WINDOW_LENGTH, NUM_LOCATION = windows.shape
HORIZON = 24

INPUT_SEQUENCE_LENGTH = WINDOW_LENGTH - HORIZON # 648
OUTPUT_SEQUENCE_LENGTH = HORIZON # 24

def get_lstm(input_seq_len: int, output_seq_len: int, num_locations: int) -> Any:
    if os.path.exists("./model/lstm.keras"):
        return load_model("./model/lstm.keras")

    units = 256

    model = Sequential([
        Input(shape=(input_seq_len, num_locations)),

        # Encoder
        LSTM(units, return_sequences=False),
        Dropout(0.2),

        # Repeat context for decoder
        RepeatVector(output_seq_len),

        # Decoder
        LSTM(units, return_sequences=True),
        Dropout(0.1),

        # Output projection
        TimeDistributed(Dense(num_locations))
    ])
    
    return model

def get_gru(input_seq_len: int, output_seq_len: int, num_locations: int) -> Any:
    if os.path.exists("./model/gru.keras"):
        return load_model("./model/gru.keras")

    units = 256

    model = Sequential([
        Input(shape=(input_seq_len, num_locations)),

        # Encoder
        GRU(units, return_sequences=False),
        Dropout(0.2),

        # Repeat context for decoder
        RepeatVector(output_seq_len),

        # Decoder
        GRU(units, return_sequences=True),
        Dropout(0.1),

        # Output projection
        TimeDistributed(Dense(num_locations))
    ])
    
    return model

def eva_regress(y_true: np.ndarray, y_pred: np.ndarray) -> None:
    """Evaluation
    evaluate the predicted resul.

    # Arguments
        y_true: List/ndarray, ture data.
        y_pred: List/ndarray, predicted data.
    """

    y_pred = y_pred.flatten()
    y_true = y_true.flatten()
    # vs = metrics.explained_variance_score(y_true, y_pred)
    # mape = metrics.mean_absolute_percentage_error(y_true, y_pred)
    mae = metrics.mean_absolute_error(y_true, y_pred)
    mse = metrics.mean_squared_error(y_true, y_pred)
    r2 = metrics.r2_score(y_true, y_pred)
    # print('explained_variance_score:%f' % vs)
    # print('mape:%f%%' % mape)
    print(f'mae: {mae}')
    print(f'mse: {mse}' % mse)
    print(f'rmse: {np.sqrt(mse)}')
    print(f'r2: {r2}')


def train_model(model: keras.Sequential, 
                X_train: np.ndarray, y_train: np.ndarray, 
                X_val: np.ndarray, y_val: np.ndarray,
                X_test: np.ndarray, y_test: np.ndarray,
                name: str, config: dict[str, Any] = {"epochs": 100, "batch_size": 32}):
    """train
    train a single model.

    # Arguments
        model: Model, NN model to train.
        X_train: ndarray(number, lags), Input data for train.
        y_train: ndarray(number, ), result data for train.
        name: String, name of model.
        config: Dict, parameter for train.
    """
    
    model.summary()
    print(f"Plotted to ./images/{name}.png")
    plot_model(
        model, to_file=f"/kaggle/working/images/{name}.png", 
        show_shapes=True, show_layer_names=True, expand_nested=True, dpi=90
    )
    model.compile(optimizer="adamw", loss="mse", metrics=['mape'])
    callbacks = [
        EarlyStopping(patience=10, restore_best_weights=True),
        ReduceLROnPlateau(patience=5, factor=0.5)
    ]
    
    history = model.fit(
        X_train, y_train,
        epochs=config.get("epochs", 0),
        batch_size=config.get("batch_size", 0),
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    model.save(f"./model/{name}.keras")
    df = pd.DataFrame.from_dict(history.history)
    df.to_csv(f"model/{name}_loss.csv", encoding='utf-8', index=False)
    # Print the final loss and validation loss
    print(f"Final Training Loss (MSE): {history.history['loss'][-1]:.4f}")
    print(f"Final Validation Loss (MSE): {history.history['val_loss'][-1]:.4f}")
    # Print the final MAE and validation MAE
    # print(f"Final Training MAE: {history.history['mae'][-1]:.4f}")
    # print(f"Final Validation MAE: {history.history['val_mae'][-1]:.4f}")
    
    
    print(f"{name.upper()} predictions:")
    y_pred = np.array(model.predict(y_test))
    assert y_pred.shape == y_test.shape, "something fishy..."
    
    # yeah they should all be normalized
    print("Predictions:")
    print("Normalized:")
    eva_regress(y_test, y_pred)
    
    print("Denomalized:")
    y_test = denormalize(scaler, y_test)
    y_pred = denormalize(scaler, y_pred)
    eva_regress(y_test, y_pred)


import sys 

def main(argv):
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--model",
        default="lstm",
        choices=["lstm", "gru"],
        help="Which model architecture to train."
    )
    parser.add_argument(
        "--epochs",
        type=int,
        default=100,
        help="Number of training epochs."
    )
    parser.add_argument(
        "--batch_size",
        type=int,
        default=32,
        help="Training batch size."
    )
    args = parser.parse_args()

    # select model
    if args.model == "lstm":
        model = get_lstm(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)
    else:
        model = get_lstm(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)

    config = {
        "epochs": args.epochs,
        "batch_size": args.batch_size
    }

    train_model(
        model,
        X_train_n, y_train_n,
        X_val_n, y_val_n,
        X_test_n, y_test_n,
        args.model,
        config
    )

config = { "epochs": 60, "batch_size": 32}
lstm = get_lstm(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)
gru = get_gru(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)
train_model(lstm, X_train_n, y_train_n, X_val_n, y_val_n, X_test_n, y_test_n, "lstm", config)
train_model(gru, X_train_n, y_train_n, X_val_n, y_val_n, X_test_n, y_test_n, "gru", config)

2025-11-24 06:53:50.292297: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763967230.485090      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763967230.541990      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 256)            │       303,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 24, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 24, 256)        │       394,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 24, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 24, 137)        │        35,209 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 733,321 (2.80 MB)

 Trainable params: 733,321 (2.80 MB)

 Non-trainable params: 0 (0.00 B)

Plotted to ./images/gru.png
Epoch 1/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 65ms/step - loss: 0.6232 - mape: 197.8624 - val_loss: 0.3125 - val_mape: 119.2462 - learning_rate: 0.0010
Epoch 2/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.2239 - mape: 150.5532 - val_loss: 0.2399 - val_mape: 107.6544 - learning_rate: 0.0010
Epoch 3/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.1825 - mape: 137.0791 - val_loss: 0.2681 - val_mape: 118.2688 - learning_rate: 0.0010
Epoch 4/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.1678 - mape: 133.1572 - val_loss: 0.2027 - val_mape: 107.3393 - learning_rate: 0.0010
Epoch 5/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.1413 - mape: 123.9228 - val_loss: 0.1904 - val_mape: 98.9728 - learning_rate: 0.0010
Epoch 6/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - loss: 0.1185 - mape: 114.1369 - val_loss: 0.2152 - val_mape: 107.6144 - learning_rate: 0.0010
Epoch 7/60
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - loss: 0.1155 - mape: 110.6887 - val_lo

In [4]:
!pip install gradio
from typing import Any
import numpy as np
import gradio as gr
import pandas as pd
from matplotlib import pyplot as plt
# from process import windows, scats
from sklearn.preprocessing import StandardScaler
# from train import get_gru, get_lstm, NUM_WINDOW, WINDOW_LENGTH, NUM_LOCATION, HORIZON, INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH
# from train import normalize, denormalize, scaler
import numpy as np
from datetime import datetime, timedelta

lstm = get_lstm(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)
gru = get_gru(INPUT_SEQUENCE_LENGTH, OUTPUT_SEQUENCE_LENGTH, NUM_LOCATION)

SCATS_TO_LOC = scats
def predict_future(model: Any, last_window: np.ndarray, scaler: StandardScaler, num_steps_ahead: int, step_size=24):
    """
    Predict future traffic values with normalization handled automatically.
    
    Args:
        model: Trained keras model.
        last_window: np.ndarray of shape (1, WINDOW_LENGTH, NUM_LOCATION)
        scaler: sklearn StandardScaler fitted on training data.
        num_steps_ahead: Total number of timesteps to predict.
        step_size: Number of timesteps to shift window each iteration.
    
    Returns:
        np.ndarray of shape (num_steps_ahead, NUM_LOCATION) with denormalized predictionsa.
    """
    # Normalize input
    current_window = normalize(scaler, last_window)
    all_predictions = []

    i = 0
    while len(all_predictions) < num_steps_ahead:
        i += 1
        pred_n = model.predict(current_window, verbose=0)  # normalized predictions
        all_predictions.append(pred_n[0])
        # print(f"Prediction #{i}: {pred_n[0]}")
        current_window = np.concatenate([
            current_window[:, step_size:, :],
            pred_n
        ], axis=1)

    predictions_n = np.vstack(all_predictions)[:num_steps_ahead]

    # Denormalize output
    predictions = denormalize(scaler, predictions_n)
    return predictions


def predict_interval(model: Any, last_window: np.ndarray, scaler: StandardScaler, 
                     scats_number: str, start_time_str: str, end_time_str: str) -> np.ndarray:
    """
    Predict for a given SCATS number and datetime interval.
    
    Args:
        model: trained keras model
        last_window: last observed window (1, WINDOW_LENGTH, NUM_LOCATION)
        scaler: fitted StandardScaler
        scats_number: str, e.g., '3002'
        start_time_str: 'YYYY-MM-DD HH:MM'
        end_time_str: 'YYYY-MM-DD HH:MM'
    
    Returns:
        np.ndarray of shape (num_steps_in_interval, num_locations_for_SCATS)
    """
    loc_indices = SCATS_TO_LOC.get(scats_number)
    if not loc_indices:
        raise ValueError("SCATS number not found!")

    start_time = pd.to_datetime(start_time_str)
    end_time = pd.to_datetime(end_time_str)
    # should predict from here
    ref_time = pd.Timestamp("2006-11-01 00:00")  # reference start

    start_step = int((start_time - ref_time) / pd.Timedelta(minutes=15))
    end_step = int((end_time - ref_time) / pd.Timedelta(minutes=15))

    total_steps_needed = end_step + 1

    # Predict
    predictions = predict_future(model, last_window, scaler, total_steps_needed)

    # Slice interval and select SCATS locations
    pred_interval = predictions[start_step:end_step + 1, loc_indices]
    # should be already flattened? Just to be sure.
    return np.array(np.median(pred_interval, axis=1)).flatten()

# ...existing code...
def predict_traffic(scats_number: str, start_time_str: str, end_time_str: str, model_choice="GRU"):
    try:
        start_time = pd.to_datetime(start_time_str)
        end_time = pd.to_datetime(end_time_str)
        if start_time.minute % 15 or end_time.minute % 15:
            raise ValueError("Timestamps must align to 15-minute intervals.")
        if start_time > end_time:
            raise ValueError("Start time must be before end time.")

        model = gru if model_choice == "GRU" else lstm
        last_window = normalize(scaler, windows[-1:])
        pred_interval = predict_interval(model, last_window, scaler, scats_number, start_time_str, end_time_str)

        timestamps = pd.date_range(start=start_time, end=end_time, freq="15min")
        if len(timestamps) != len(pred_interval):
            raise ValueError(f"Timestamp count {len(timestamps)} does not match prediction length {len(pred_interval)}.")

        text = (
            f"Predicted flow for SCATS {scats_number}\n"
            f"From {start_time} to {end_time}\n"
            f"{len(pred_interval)} intervals (15 min)\n\n"
            f"{pred_interval}"
        )

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(timestamps, pred_interval, marker="o")
        ax.set_title(f"Predicted Flow for SCATS {scats_number}")
        ax.set_xlabel("Time")
        ax.set_ylabel("Flow")
        ax.grid(True, alpha=0.3)
        fig.autofmt_xdate()

        return text, fig

    except Exception as e:
        return f"Error: {str(e)}", None


if __name__ == "__main__":
    # Create Gradio interface
    interface = gr.Interface(
    fn=predict_traffic,
    inputs=[
        gr.Textbox(label="SCATS Number", placeholder="3002"),
        gr.Textbox(label="Start Time", placeholder="2006-11-05 09:30"),
        gr.Textbox(label="End Time", placeholder="2006-11-05 11:30"),
        gr.Radio(["LSTM", "GRU"], label="Model", value="GRU")
    ],
    outputs=[
        gr.Textbox(label="Prediction Output"),
        gr.Plot(label="Flow Plot")
    ],
    title="Traffic Flow Prediction",
    description="Predict traffic flow for a given SCATS location and time interval")

interface.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 30.9 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.4
    Uninstalling pydantic-2.12.4:
      Successfully uninstalled pydantic-2.12.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://81363219f87f4b33b5.gradio.live

This share link expires in 1 week. For 